# Exploración de Dispersión de Precios por SKU

Este notebook evalúa la dispersión del precio unitario a nivel de `cliente padre` por `Mes`, `Marca`, `SKU` y `Zona`, con métricas robustas y ponderadas por volumen.

## 1. Configuración y Carga de Datos

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid')
pd.options.display.float_format = '{:,.2f}'.format

In [2]:
PATH_DATOS = 'datos/BBDD Ventas 25 - 26.xlsx'

# Opcional: activar filtro de marcas
USAR_FILTRO_MARCAS = False
MARCAS_FILTRO = ['LA PREFERIDA', 'SAN JORGE', 'WINTER']

# Parámetros del análisis robusto
WINSOR_P_LOW = 0
WINSOR_P_HIGH = 1
MIN_OBS_RANK = 8
MIN_MESES_TEMPORAL = 3

In [3]:
df_raw = pd.read_excel(PATH_DATOS, header=1)
columnas = [col for col in df_raw.columns if 'Unnamed' not in str(col)]

print(f'Filas: {df_raw.shape[0]:,} | Columnas: {df_raw.shape[1]:,}')
print(f'Columnas útiles detectadas: {len(columnas)}')

Filas: 434,066 | Columnas: 44
Columnas útiles detectadas: 43


## 2. Selección y Preparación de Variables

In [14]:
columnas_importantes = [
    'Mes',
    'SKU', 'Descripción', 'MARCA', 'FAMILIA HANA', 'TIPO CARNE',
    'CANAL HANA', 'ZONAL HANA', 'COD. CLIENTE PADRE',
    'KILOS',
    'VENTAS'
]

df_modelo = df_raw[columnas_importantes].copy()
df_modelo.info()

<class 'pandas.DataFrame'>
RangeIndex: 434066 entries, 0 to 434065
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Mes                 434066 non-null  object 
 1   SKU                 434066 non-null  int64  
 2   Descripción         434066 non-null  str    
 3   MARCA               434066 non-null  str    
 4   FAMILIA HANA        434066 non-null  str    
 5   TIPO CARNE          433952 non-null  str    
 6   CANAL HANA          434066 non-null  str    
 7   ZONAL HANA          434066 non-null  str    
 8   COD. CLIENTE PADRE  282820 non-null  float64
 9   KILOS               434066 non-null  float64
 10  VENTAS              434066 non-null  int64  
dtypes: float64(2), int64(2), object(1), str(6)
memory usage: 36.4+ MB


In [15]:
# Normalizar Mes: mezcla de datetime + serial de fecha Excel
mes_num = pd.to_numeric(df_modelo['Mes'], errors='coerce')
mask_excel = mes_num.notna()
mes_dt = pd.to_datetime(df_modelo['Mes'].where(~mask_excel), errors='coerce')
mes_dt.loc[mask_excel] = pd.to_datetime(mes_num.loc[mask_excel], unit='D', origin='1899-12-30')
df_modelo['Mes'] = mes_dt

# Estandarizar texto para evitar duplicados por formato
for col in ['MARCA', 'FAMILIA HANA', 'CANAL HANA', 'ZONAL HANA']:
    df_modelo[col] = df_modelo[col].astype(str).str.strip().str.upper()

# SKU y cliente padre como enteros nullable
df_modelo['SKU'] = pd.to_numeric(df_modelo['SKU'], errors='coerce').astype('Int64')
df_modelo['COD. CLIENTE PADRE'] = pd.to_numeric(df_modelo['COD. CLIENTE PADRE'], errors='coerce').astype('Int64')

# Mantener filas con mes y SKU válidos
df_modelo = df_modelo.dropna(subset=['Mes', 'SKU']).copy()

# Filtrar para tener sólo filas con COD. CLIENTE PADRE no vacío
df_modelo = df_modelo.dropna(subset=['COD. CLIENTE PADRE']).copy()

if USAR_FILTRO_MARCAS:
    df_modelo = df_modelo[df_modelo['MARCA'].isin(MARCAS_FILTRO)].copy()

print(f'Filas tras limpieza: {df_modelo.shape[0]:,}')

Filas tras limpieza: 282,820


In [16]:
sku_catalogo = (
    df_modelo[['SKU', 'Descripción']]
    .dropna(subset=['SKU'])
    .drop_duplicates()
    .groupby('SKU', as_index=False)
    .agg(DESCRIPCION_SKU=('Descripción', 'first'))
)

print(sku_catalogo.shape[0], 'SKUs únicos en catálogo')
df_modelo.head()

268 SKUs únicos en catálogo


,Mes,SKU,Descripción,MARCA,FAMILIA HANA,TIPO CARNE,CANAL HANA,ZONAL HANA,COD. CLIENTE PADRE,KILOS,VENTAS
3,2025-01-01,20,ARROLLADO LOMO CON AJI SJ,SAN JORGE,ARROLLADOS,CERDO,TR,ARICA,1196346,34.55,164684
7,2025-01-01,20,ARROLLADO LOMO CON AJI SJ,SAN JORGE,ARROLLADOS,CERDO,TR,CHILLAN,1205327,120.95,594246
11,2025-01-01,20,ARROLLADO LOMO CON AJI SJ,SAN JORGE,ARROLLADOS,CERDO,TR,CONCEPCION,1010880,8.35,39793
12,2025-01-01,20,ARROLLADO LOMO CON AJI SJ,SAN JORGE,ARROLLADOS,CERDO,TR,CONCEPCION,1013028,8.08,38507
13,2025-01-01,20,ARROLLADO LOMO CON AJI SJ,SAN JORGE,ARROLLADOS,CERDO,TR,CONCEPCION,91995,73.20,348844


## 3. Construcción de Dataset Analítico (Nivel Cliente Padre por SKU)

In [17]:
keys_nivel = ['Mes', 'MARCA', 'SKU', 'ZONAL HANA', 'COD. CLIENTE PADRE']

df_final = (
    df_modelo.groupby(keys_nivel, as_index=False, dropna=False)
    .agg(
        VENTAS=('VENTAS', 'sum'),
        KILOS_VENDIDOS=('KILOS', 'sum')
    )
)

df_final['Precio Unitario Promedio'] = np.where(
    df_final['KILOS_VENDIDOS'] > 0,
    df_final['VENTAS'] / df_final['KILOS_VENDIDOS'],
    np.nan
)

# Evitar divisiones inválidas
df_final = df_final[df_final['KILOS_VENDIDOS'] > 0].copy()

print(f'Observaciones analíticas: {df_final.shape[0]:,}')

Observaciones analíticas: 276,243


In [18]:
df_final.head()

,Mes,MARCA,SKU,ZONAL HANA,COD. CLIENTE PADRE,VENTAS,KILOS_VENDIDOS,Precio Unitario Promedio
0,2025-01-01,LA PREFERIDA,3001,ANTOFAGASTA,61905,254717,39.14,"6,507.84"
1,2025-01-01,LA PREFERIDA,3001,ANTOFAGASTA,62332,3779006,586.35,"6,444.98"
2,2025-01-01,LA PREFERIDA,3001,ARICA,61905,256070,39.84,"6,428.27"
3,2025-01-01,LA PREFERIDA,3001,ARICA,62332,290356,46.23,"6,280.00"
4,2025-01-01,LA PREFERIDA,3001,CALAMA,61905,427873,65.92,"6,491.28"


## 4. Winsorización y Helpers de Métricas

In [19]:
winsor_group = ['ZONAL HANA', 'MARCA', 'SKU']

def weighted_mean(values, weights):
    v = np.asarray(values, dtype=float)
    w = np.asarray(weights, dtype=float)
    m = np.isfinite(v) & np.isfinite(w) & (w > 0)
    if m.sum() == 0:
        return np.nan
    return np.average(v[m], weights=w[m])

def weighted_std(values, weights):
    v = np.asarray(values, dtype=float)
    w = np.asarray(weights, dtype=float)
    m = np.isfinite(v) & np.isfinite(w) & (w > 0)
    if m.sum() == 0:
        return np.nan
    v = v[m]
    w = w[m]
    mu = np.average(v, weights=w)
    return np.sqrt(np.average((v - mu) ** 2, weights=w))

def iqr(values):
    s = pd.Series(values).dropna()
    if s.empty:
        return np.nan
    return s.quantile(0.75) - s.quantile(0.25)

def mad(values):
    s = pd.Series(values).dropna()
    if s.empty:
        return np.nan
    med = s.median()
    return np.median(np.abs(s - med))

In [20]:
df_winsor = df_final.copy()

df_winsor['p_low'] = df_winsor.groupby(winsor_group)['Precio Unitario Promedio'].transform(
    lambda s: s.quantile(WINSOR_P_LOW)
)
df_winsor['p_high'] = df_winsor.groupby(winsor_group)['Precio Unitario Promedio'].transform(
    lambda s: s.quantile(WINSOR_P_HIGH)
)
df_winsor['Precio Unitario Winsor'] = df_winsor['Precio Unitario Promedio'].clip(
    lower=df_winsor['p_low'],
    upper=df_winsor['p_high']
)

print(f'Observaciones tras winsorización: {df_winsor.shape[0]:,}')

Observaciones tras winsorización: 276,243


# 5. Dispersión de precio por SKU

## 5. Dispersión de precio de SKU por Zona

In [21]:
def agg_dispersion_global(g):
    p = g['Precio Unitario Winsor']
    w = g['KILOS_VENDIDOS']
    return pd.Series({
        'n_observaciones': p.count(),
        'mean_w': weighted_mean(p, w),
        'std_w': weighted_std(p, w),
        'mediana': p.median(),
        'iqr': iqr(p),
        'mad': mad(p)
    })

dispersion_precio = (
    df_winsor.groupby(['ZONAL HANA', 'SKU'], as_index=False)
    .apply(agg_dispersion_global, include_groups=False)
    .reset_index()
)
if 'level_3' in dispersion_precio.columns:
    dispersion_precio = dispersion_precio.drop(columns=['level_3'])

dispersion_precio = dispersion_precio[dispersion_precio['n_observaciones'] >= MIN_OBS_RANK].copy()
dispersion_precio = dispersion_precio.sort_values(['std_w', 'n_observaciones'], ascending=[False, False])
dispersion_precio = dispersion_precio.merge(sku_catalogo, on='SKU', how='left')

print('Top dispersión global robusta por SKU:')
dispersion_precio.head(20)

Top dispersión global robusta por SKU:


,index,ZONAL HANA,SKU,n_observaciones,mean_w,std_w,mediana,iqr,mad,DESCRIPCION_SKU
0,3336,SANTIAGO,3945,27.00,"48,036.02","34,996.97","47,026.67","8,240.93","1,454.33",CAJA SNACKIN BEEF JERKY LP 10x20 G.
1,137,ANTOFAGASTA,3879,33.00,"15,332.82","12,728.17","14,019.97","4,427.19",3.02,SNACKIN SALAME LP 22x18 G.
2,4031,VIÑA DEL MAR,3945,8.00,"45,034.66","11,516.80","48,480.00","9,020.44","6,857.61",CAJA SNACKIN BEEF JERKY LP 10x20 G.
3,2033,OSORNO,3945,8.00,"48,610.98","11,124.29","47,022.71","25,084.18","11,507.01",CAJA SNACKIN BEEF JERKY LP 10x20 G.
4,3798,TEMUCO,3945,11.00,"53,358.41","9,509.26","48,480.00","11,261.37","10,048.98",CAJA SNACKIN BEEF JERKY LP 10x20 G.
5,1603,LA SERENA,3945,8.00,"47,095.77","8,649.27","48,480.16",937.86,0.75,CAJA SNACKIN BEEF JERKY LP 10x20 G.
6,3101,SAN FERNANDO,3945,10.00,"46,575.37","8,553.56","47,026.83","3,724.29",727.50,CAJA SNACKIN BEEF JERKY LP 10x20 G.
7,3339,SANTIAGO,3951,245.00,"24,288.36","7,089.44","27,061.16","3,563.00","1,980.80",JAMON SERRANO 14x80 GR LP
8,1840,LOS ANGELES,3998,19.00,"21,872.38","6,312.92","17,362.96","7,092.59","1,773.03",ESTUCHE SNACKIN SPICY LP 6x18 G.
9,4050,VIÑA DEL MAR,3996,30.00,"15,443.83","5,614.72","19,166.87","3,017.91","1,283.54",CAJA SNACKIN SPICY LP 10x18 G.


## 5.1 Dispersión de precio de SKU sin Zona

In [22]:
dispersion_precio_sin_zona_all = (
    df_winsor.groupby(['SKU'], as_index=False)
    .apply(agg_dispersion_global, include_groups=False)
    .reset_index()
)

for col in ['level_1', 'level_2', 'level_3', 'level_4']:
    if col in dispersion_precio_sin_zona_all.columns:
        dispersion_precio_sin_zona_all = dispersion_precio_sin_zona_all.drop(columns=[col])

dispersion_precio_sin_zona = (
    dispersion_precio_sin_zona_all
    .query('n_observaciones >= @MIN_OBS_RANK')
    .sort_values(['std_w', 'n_observaciones'], ascending=[False, False])
    .merge(sku_catalogo, on='SKU', how='left')
)

cols_top = ['SKU', 'DESCRIPCION_SKU', 'n_observaciones', 'mean_w', 'std_w', 'mediana', 'iqr', 'mad']
dispersion_precio_sin_zona[cols_top].head(20)

,SKU,DESCRIPCION_SKU,n_observaciones,mean_w,std_w,mediana,iqr,mad
0,3945,CAJA SNACKIN BEEF JERKY LP 10x20 G.,127.00,"46,744.33","28,033.61","48,480.00","25,083.87","10,049.77"
1,3951,JAMON SERRANO 14x80 GR LP,"1,303.00","26,227.61","5,638.22","29,337.95","2,689.29",367.19
2,3996,CAJA SNACKIN SPICY LP 10x18 G.,303.00,"19,778.92","3,577.96","19,167.01","3,017.67","1,283.68"
3,3998,ESTUCHE SNACKIN SPICY LP 6x18 G.,302.00,"19,737.17","3,457.78","17,363.01","6,039.70","1,773.20"
4,3997,CAJA SNACKIN SALAME LP 10x18 G.,388.00,"20,882.21","3,286.30","20,900.00","3,660.22","1,733.33"
5,3879,SNACKIN SALAME LP 22x18 G.,"1,162.00","15,613.48","3,253.95","17,884.03","6,882.57","3,017.48"
6,3994,CAJA SNACKIN SPICY LP 22x18 G.,409.00,"15,830.80","2,943.15","17,705.00","6,883.33","3,685.14"
7,3992,SNACKIN SALAME LP 10x18 G.,506.00,"17,644.89","2,922.28","17,497.92","5,484.68","1,279.08"
8,3050,PECHUGA COCIDA PAVO CT 10x125 G. LP,145.00,"10,830.55","2,736.98","12,659.04","2,353.10","1,647.14"
9,3323,LOMO KASSLER LP 100 G.,237.00,"11,345.49","2,392.75","12,481.27","2,647.00","2,165.73"
